# TMDb Movie Analysis

TMDb discover/movie API로 받아온 데이터를 pandas로 분석합니다.

## 사전 준비: TMDb API 키 발급

TMDb API를 사용하려면 TMDb 계정으로 로그인한 뒤 API 설정에서 키를 신청해야 합니다.

### 1. TMDb 로그인

[TMDb 공식 사이트](https://www.themoviedb.org/) 에 접속해 계정을 만들거나 로그인합니다.

### 2. API 설정으로 이동

로그인 후 프로필 아이콘 클릭 → **설정(Settings)** → **API** 메뉴를 엽니다.

### 3. API 키 신청

* **Request an API Key** 또는 해당 링크를 클릭합니다.
* 사용 유형에서 **Developer** 를 선택합니다.
* TMDb API 이용약관을 확인하고 동의합니다.

### 4. 애플리케이션 정보 입력

| 항목 | 설명 | 예시 |
|---|---|---|
| Application Name | 애플리케이션 이름 | My Movie App |
| Application URL | 프로젝트 또는 서비스 URL | 본인 프로젝트 주소 |
| Application Summary | API를 사용하는 목적 | 개인 학습용 영화 목록 조회 애플리케이션 |
| Contact Information | 연락처 정보 | 본인 이메일 |

실제 사용 목적과 연락처를 정확하게 입력한 뒤 제출합니다.

### 5. 키 확인

신청이 완료되면 API 설정 화면에서 **API 자격 증명(API Credentials)** 을 확인할 수 있습니다.

### 6. 코드에 적용

발급받은 API 키를 아래 코드의 `TMDB_API_KEY` 값에 입력 후 실행합니다.

In [ ]:
import requests
import pandas as pd
pd.set_option('display.max_rows', None)

url = "https://api.themoviedb.org/3/discover/movie"

response = requests.get(
    url,
    params={"api_key": "<YOUR_API_KEY>"}
)

df = pd.json_normalize(response.json()["results"])

## 3. TMDb 데이터 pandas 분석 (5가지 주제)

`cell 85`에서 받아온 `df`(TMDb discover/movie)를 활용한 간단한 분석입니다.

| # | 분석 주제 | 활용 컬럼 |
|---|---|---|
| 1 | 출시 연도별 영화 개수 | `release_date` |
| 2 | 평점 구간별 영화 개수 분포 | `vote_average` |
| 3 | 원어(original_language)별 영화 개수 | `original_language` |
| 4 | 인기도(popularity) Top 10 영화 | `popularity`, `title` |
| 5 | 출시 월별 영화 개수 | `release_date` |

In [2]:
# 주제 1: 출시 연도별 영화 개수
# release_date에서 연도를 추출하여 연도별 영화 개수 집계
df['year'] = pd.to_datetime(df['release_date']).dt.year

year_counts = df['year'].value_counts().sort_index()
display(year_counts)

year
1984     1
2008     1
2017     1
2026    17
Name: count, dtype: int64

In [3]:
# 주제 2: 평점 구간별 영화 개수 분포
# vote_average를 1점 단위 구간으로 나누어 개수 집계
bins = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
labels = ['0~1', '1~2', '2~3', '3~4', '4~5', '5~6', '6~7', '7~8', '8~9', '9~10']
df['vote_bucket'] = pd.cut(df['vote_average'], bins=bins, labels=labels, right=False)

vote_dist = df['vote_bucket'].value_counts().sort_index()
display(vote_dist)

vote_bucket
0~1     0
1~2     0
2~3     0
3~4     0
4~5     0
5~6     2
6~7     6
7~8     6
8~9     6
9~10    0
Name: count, dtype: int64

In [4]:
# 주제 3: 원어(original_language)별 영화 개수
lang_counts = df['original_language'].value_counts()
display(lang_counts)

original_language
en    13
es     3
ko     1
ja     1
hi     1
de     1
Name: count, dtype: int64

In [5]:
# 주제 4: 인기도(popularity) Top 10 영화
top10 = df.nlargest(10, 'popularity')[['title', 'popularity', 'vote_average', 'release_date']]
top10 = top10.reset_index(drop=True)
top10.index = top10.index + 1  # 1부터 시작하는 순위
display(top10)

,title,popularity,vote_average,release_date
1,Project Hail Mary,1071.9478,8.600,2026-03-15
2,Resident Evil,624.6254,7.500,2026-09-16
3,Heart of the Beast,554.0784,7.500,2026-09-19
4,The Odyssey,456.7546,8.000,2026-07-15
5,Spider-Man: Brand New Day,357.4740,8.473,2026-07-29
6,The Dark Knight,328.5126,8.540,2008-07-16
7,Hope,316.9294,6.761,2026-07-15
8,Apocalypse Z: New World,299.9329,6.247,2026-10-09
9,A Story of Derry,296.4494,6.757,2026-10-31
10,Other Mommy,226.3965,5.883,2026-10-07


In [6]:
# 주제 5: 출시 월별 영화 개수
# release_date에서 월을 추출하여 월별 영화 개수 집계
df['month'] = pd.to_datetime(df['release_date']).dt.month

month_counts = df['month'].value_counts().sort_index()
display(month_counts)

month
3     2
7     4
9     4
10    8
11    2
Name: count, dtype: int64